## Image processing - food classifier

### Part I - Baseline Evaluation

Using our own collected data as the input to the classification model. Compare the results versus the results from the Food-101 dataset

First test if the model correctly loaded, and predict example image

In [ ]:
import shutil
import sys
sys.path.insert(0, './src')
from models.classifier import FoodClassifier
from utils import calculate_metrics
import plot

# Initialize the classification model
model = FoodClassifier()

# Model testing, predict a single image
example_image = './data/example.JPG'
results = model.predict_single(example_image)
plot.single_image(example_image, title=f"{results['label']}: {results['confidence']:.2f}")

Run prediction on our collected dataset as the baseline

In [ ]:
dataset_path = './data/raw'
results = model.predict_folder(dataset_path)
plot.wrong_predictions(results, folder = dataset_path)
metrics = calculate_metrics(**results)
print(metrics)

### Part II - Individual Algorithm Evaluation

Process our collected data with the image processing algorithms. Use them as the input to the model. Find which individual algorithm performs best

In [ ]:
from preprocessing.lowlight import gamma, CLAHE, SSRetinex
from preprocessing.deblurr import ssk, usm, swf
#form preprocessing.downscale import d1, d2, d3
from utils import preprocess_folder

output_path = './data/preprocessed'

# Low-light
for func in [gamma, CLAHE, SSRetinex]:
    preprocess_folder(func, input_dir=dataset_path, output_dir=output_path)
    results = model.predict_folder(output_path)
    metrics = calculate_metrics(**results)
    print(f"Metrics for {func.__name__}: {metrics}")
    #plot.wrong_predictions(results, folder = output_path)
shutil.rmtree(output_path)

# Deblurring
for func in [ssk, usm, swf]:
    preprocess_folder(func, input_dir=dataset_path, output_dir=output_path)
    results = model.predict_folder(output_path)
    metrics = calculate_metrics(**results)
    print(f"Metrics for {func.__name__}: {metrics}")
    #plot.wrong_predictions(results, folder = output_path)
shutil.rmtree(output_path)

# Downscaling
# for func in [d1, d2, d3]:

### Part III - Combination Search

Find the best low-light - deblur - downscale combination

In [ ]:

# find the best low-light - deblur - downscale combination

output_path = './data/preprocessed'
best_metrics = None
best_combination = None

for func1 in [gamma, CLAHE, SSRetinex]:
    for func2 in [ssk, usm, swf]:
        #for func3 in [d1, d2, d3]:
            preprocess_folder(func1, input_dir=dataset_path, output_dir=output_path)
            preprocess_folder(func2, input_dir=output_path, output_dir=output_path)
            #preprocess_folder(func3, input_dir=output_path, output_dir=output_path)
            results = model.predict_folder(output_path)
            metrics = calculate_metrics(**results)
            print(f"Metrics for {func1.__name__} + {func2.__name__}: {metrics}")
            #plot.wrong_predictions(results, folder = output_path)
            shutil.rmtree(output_path)

            if best_metrics is None or metrics['accuracy'] > best_metrics['accuracy']:
                best_metrics = metrics
                best_combination = (func1.__name__, func2.__name__) #, func3.__name__)

print(f"Best combination: {best_combination} with metrics: {best_metrics}")

### Part IV - Model Fine-Tuning

In [ ]:
from models.trainer import fine_tune

fine_tune(
    model_wrapper=model,
    data_dir=dataset_path,
    val_ratio=0.3,
    epochs=20,
    batch_size=4,
    lr=5e-5
)